# Carry & Vol Regime Explorer

Loads accumulated Deribit Parquet history, computes rolling carry / vol
divergence, and visualises regime transitions for BTC.

## 1 — Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from basis_analytics import (
    annualized_carry,
    annualized_funding,
    atm_term_structure,
    basis_curve,
    carry_vol_divergence,
    close_to_close_rv,
    percentile_rank,
)

plt.rcParams.update(
    {"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9}
)

In [ ]:
parquet_dir = Path("data/parquet/deribit")
frames = [pd.read_parquet(p) for p in sorted(parquet_dir.glob("*/tickers.parquet"))]
df = pd.concat(frames, ignore_index=True)

# Focus on BTC
btc = df[df.currency == "BTC"].copy()

perps = btc[btc.asset_kind == "perpetual"].copy()
futures = btc[
    (btc.asset_kind == "future") & (~btc.symbol.str.contains("PERPETUAL"))
].copy()
options = btc[btc.asset_kind == "option"].copy()

timestamps = sorted(btc.timestamp.unique())
print(f"Loaded {len(df):,} rows  ({len(btc):,} BTC)")
print(
    f"Date range: {pd.Timestamp(timestamps[0]):%Y-%m-%d}"
    f" \u2192 {pd.Timestamp(timestamps[-1]):%Y-%m-%d}"
)
print(f"Snapshots:  {len(timestamps)}")
print(
    f"Futures:    {futures.symbol.nunique()} contracts"
    f"  |  Perps: {perps.symbol.nunique()}"
)
print(f"Options:    {options.shape[0]:,} rows")

## 2 — Futures Carry Time-Series

For each snapshot we compute `annualized_carry(spot, future, dte)` for
every BTC dated future, using the BTC-PERPETUAL mark price as the spot
proxy.  Futures are bucketed into near (&lt;30 d), mid (30–90 d), and far
(&gt;90 d) tenors.

In [ ]:
records = []
for ts in timestamps:
    snap_perp = perps[perps.timestamp == ts]
    snap_fut = futures[futures.timestamp == ts]
    if snap_perp.empty:
        continue
    spot = snap_perp.iloc[0].mark_price
    for _, row in snap_fut.iterrows():
        dte = (row.expiry - row.timestamp).total_seconds() / 86400
        if dte <= 0:
            continue
        carry = annualized_carry(spot, row.mark_price, dte)
        records.append(
            {"timestamp": ts, "symbol": row.symbol, "dte": dte, "carry": carry}
        )

carry_df = pd.DataFrame(records)
carry_df["tenor"] = pd.cut(
    carry_df.dte,
    bins=[0, 30, 90, 1e4],
    labels=["Near (<30d)", "Mid (30-90d)", "Far (>90d)"],
)
print(
    f"{len(carry_df)} carry observations across {carry_df.symbol.nunique()} contracts"
)
carry_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for tenor, grp in carry_df.groupby("tenor", observed=True):
    med = grp.groupby("timestamp")["carry"].median().sort_index()
    ax.plot(med.index, med.values * 100, marker=".", markersize=4, label=tenor)
ax.set_ylabel("Annualised carry (%)")
ax.set_title("BTC Futures Carry by Tenor Bucket (median per snapshot)")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3 — Basis Curve Snapshots

Three representative dates (start, middle, end of the dataset) show how
the term structure of carry evolves.

In [ ]:
sample_ts = [timestamps[0], timestamps[len(timestamps) // 2], timestamps[-1]]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, ts in zip(axes, sample_ts, strict=True):
    snap_perp = perps[perps.timestamp == ts]
    snap_fut = futures[futures.timestamp == ts]
    if snap_perp.empty or snap_fut.empty:
        ax.set_title(f"{pd.Timestamp(ts):%Y-%m-%d}\n(no data)")
        continue
    spot = snap_perp.iloc[0].mark_price
    curve = basis_curve(
        snap_fut.mark_price.tolist(),
        snap_fut.expiry.tolist(),
        spot=spot,
        asof=pd.Timestamp(ts).to_pydatetime(),
    )
    ax.bar(curve.days, curve.annualized_carry * 100, width=3, color="steelblue")
    ax.set_xlabel("Days to expiry")
    ax.set_title(f"{pd.Timestamp(ts):%Y-%m-%d %H:%M}")
axes[0].set_ylabel("Annualised carry (%)")
fig.suptitle("BTC Basis Curve — 3 Snapshots", fontsize=11)
plt.tight_layout()
plt.show()

## 4 — Perpetual Funding

Extract the 8-hour funding rate from BTC-PERPETUAL and annualise it
with `annualized_funding()`.

In [ ]:
funding_raw = (
    perps[["timestamp", "funding_rate"]]
    .dropna(subset=["funding_rate"])
    .drop_duplicates(subset=["timestamp"])
    .set_index("timestamp")
    .sort_index()
    .squeeze()
)
ann_funding = annualized_funding(funding_raw)

print(f"Funding observations: {len(funding_raw)}")
print(f"8h rate  — mean: {funding_raw.mean():.6f},  std: {funding_raw.std():.6f}")
print(f"Annual   — mean: {ann_funding.mean():.4f},  std: {ann_funding.std():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].bar(funding_raw.index, funding_raw.values * 100, width=0.3, color="tab:purple")
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("8h rate (%)")
axes[0].set_title("BTC-PERPETUAL Funding Rate")

axes[1].plot(
    ann_funding.index,
    ann_funding.values * 100,
    color="tab:orange",
    marker=".",
    markersize=4,
)
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_ylabel("Annualised (%)")
axes[1].set_title("Annualised Funding Rate")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5 — Carry vs Funding Divergence

Align the median near-term futures carry with the annualised perp
funding rate on a common timeline and plot their divergence.

In [ ]:
near = carry_df[carry_df.tenor == "Near (<30d)"]
carry_ts = (
    near.groupby("timestamp")["carry"].median().sort_index().rename("futures_carry")
)

# Align on common timestamps
aligned = pd.DataFrame(
    {"futures_carry": carry_ts, "perp_funding": ann_funding}
).dropna()
aligned["divergence"] = aligned.futures_carry - aligned.perp_funding

print(f"Aligned points: {len(aligned)}")
print(aligned.describe().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = ["tab:green" if v >= 0 else "tab:red" for v in aligned.divergence]
ax.bar(aligned.index, aligned.divergence * 100, width=0.3, color=colors)
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("Carry \u2212 Funding (%)")
ax.set_title("BTC Carry vs Funding Divergence (near-term futures \u2212 perp)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6 — ATM IV vs Realized Vol

For each snapshot we compute ATM IV via `atm_term_structure()` (picking
the nearest-to-30-day expiry) and compare it against close-to-close
realised volatility from BTC-PERPETUAL mark prices.

In [ ]:
# Build a futures forward lookup: (timestamp, expiry) -> mark_price
fwd_lookup = futures.set_index(["timestamp", "expiry"])["mark_price"].to_dict()

atm_records = []
for ts in timestamps:
    snap_opt = options[options.timestamp == ts].copy()
    if snap_opt.empty:
        continue
    # Map forward price from the futures with matching expiry
    snap_opt["forward"] = snap_opt.expiry.map(
        lambda exp, _ts=ts: fwd_lookup.get((_ts, exp), np.nan)
    )
    snap_opt = snap_opt.dropna(subset=["forward", "mark_iv"])
    if snap_opt.empty:
        continue
    atm = atm_term_structure(snap_opt[["expiry", "strike", "forward", "mark_iv"]])
    if atm.empty:
        continue
    # Pick the expiry closest to 30 days from now
    dte = (atm.index - ts).total_seconds() / 86400
    mask = dte > 0
    if not mask.any():
        continue
    closest_idx = int(np.abs(dte[mask] - 30).argmin())
    row = atm[mask].iloc[closest_idx]
    atm_records.append({"timestamp": ts, "atm_iv": row.mark_iv})

atm_iv = pd.DataFrame(atm_records).set_index("timestamp").sort_index().squeeze()
print(f"ATM IV observations: {len(atm_iv)}")

In [ ]:
# RV from perp mark_price (shorter window for sparse data)
perp_prices = (
    perps[["timestamp", "mark_price"]]
    .drop_duplicates(subset=["timestamp"])
    .set_index("timestamp")
    .sort_index()
    .squeeze()
)
rv = close_to_close_rv(perp_prices, window="14D").dropna()

print(f"RV observations: {len(rv)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    atm_iv.index,
    atm_iv.values * 100,
    label="ATM IV (~30d)",
    marker=".",
    markersize=4,
)
ax.plot(
    rv.index,
    rv.values * 100,
    label="Realised Vol (14d)",
    marker=".",
    markersize=4,
)
ax.fill_between(
    atm_iv.index,
    atm_iv.values * 100,
    np.interp(
        atm_iv.index.astype(np.int64),
        rv.index.astype(np.int64),
        rv.values * 100,
    ),
    alpha=0.15,
    color="orange",
    label="IV \u2212 RV spread",
)
ax.set_ylabel("Volatility (%)")
ax.set_title("BTC ATM IV vs Realised Volatility")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7 — Rolling Percentile Signals

With only ~32 days of data we use a 30-day lookback for `percentile_rank()`.
`carry_vol_divergence()` maps two percentile series to a score in
[-1, 1] where positive = carry rich relative to vol.

In [ ]:
# Median near-term carry series (reindex to DatetimeIndex)
carry_series = carry_ts.copy()
carry_series.index = pd.DatetimeIndex(carry_series.index)

# IV - RV spread
iv_rv_spread = pd.Series(
    atm_iv.values
    - np.interp(atm_iv.index.astype(np.int64), rv.index.astype(np.int64), rv.values),
    index=atm_iv.index,
    name="iv_rv_spread",
)

# Percentile ranks (30-day window for our short dataset)
carry_pctile = percentile_rank(carry_series, window="30D")
iv_rv_pctile = percentile_rank(iv_rv_spread, window="30D")
divergence_score = carry_vol_divergence(carry_pctile, iv_rv_pctile)

print(
    f"Carry percentile range: {carry_pctile.min():.2f} \u2192 {carry_pctile.max():.2f}"
)
print(
    f"IV-RV percentile range: {iv_rv_pctile.min():.2f} \u2192 {iv_rv_pctile.max():.2f}"
)
print(
    f"Divergence score range: {divergence_score.min():.2f}"
    f" \u2192 {divergence_score.max():.2f}"
)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(
    carry_pctile.index,
    carry_pctile.values,
    label="Carry pctile",
    color="steelblue",
)
axes[0].plot(
    iv_rv_pctile.index,
    iv_rv_pctile.values,
    label="IV\u2212RV pctile",
    color="darkorange",
)
axes[0].axhline(0.8, ls="--", color="red", lw=0.8, label="Elevated (0.8)")
axes[0].axhline(0.2, ls="--", color="green", lw=0.8, label="Depressed (0.2)")
axes[0].set_ylabel("Percentile")
axes[0].set_title("Rolling Percentile Ranks (30-day window)")
axes[0].legend(fontsize=8)

axes[1].plot(
    divergence_score.index,
    divergence_score.values,
    color="purple",
    marker=".",
    markersize=4,
)
axes[1].axhline(0, color="k", lw=0.5)
axes[1].axhspan(0.5, 1.0, alpha=0.08, color="red", label="Carry-rich")
axes[1].axhspan(-1.0, -0.5, alpha=0.08, color="green", label="Vol-rich")
axes[1].set_ylabel("Score [-1, 1]")
axes[1].set_title("Carry\u2013Vol Divergence Score")
axes[1].legend(fontsize=8)

# Regime zones annotated on carry
axes[2].plot(
    carry_pctile.index,
    carry_series.reindex(carry_pctile.index).values * 100,
    color="steelblue",
    marker=".",
    markersize=4,
)
for ts_val, pct in carry_pctile.items():
    if pct >= 0.8:
        axes[2].axvline(ts_val, color="red", alpha=0.15, lw=3)
    elif pct <= 0.2:
        axes[2].axvline(ts_val, color="green", alpha=0.15, lw=3)
axes[2].set_ylabel("Carry (%)")
axes[2].set_title(
    "Near-Term Carry with Regime Highlights (red=elevated, green=depressed)"
)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 8 — Summary

In [ ]:
summary = pd.DataFrame(
    {
        "Metric": [
            "Near-term carry (median, ann %)",
            "Perp funding (ann %)",
            "ATM IV (~30d, %)",
            "Realised vol (14d, %)",
            "IV \u2212 RV spread (pp)",
            "Divergence score (mean)",
            "Snapshots analysed",
        ],
        "Value": [
            f"{carry_series.median() * 100:.2f}",
            f"{ann_funding.median() * 100:.2f}",
            f"{atm_iv.median() * 100:.1f}",
            f"{rv.median() * 100:.1f}",
            f"{iv_rv_spread.median() * 100:.1f}",
            f"{divergence_score.mean():.3f}",
            f"{len(timestamps)}",
        ],
    }
)
print(summary.to_string(index=False))

### Interpretation Guide

| Divergence score | Regime | Meaning |
|:---:|:---:|:---|
| > 0.5 | **Carry-rich** | Futures basis elevated relative to vol premium — potential mean-reversion short carry |
| −0.5 to 0.5 | **Neutral** | No strong signal |
| < −0.5 | **Vol-rich** | Vol premium elevated relative to carry — consider long gamma / short vol trades |

> **Note:** With only ~32 days of data the percentile ranks have limited
> statistical power.  As more snapshots accumulate the signals will
> become more meaningful.